# Notebook 4: Gradio demo

In [2]:
import gradio as gr
from sentence_transformers import SentenceTransformer
from chromadb import PersistentClient
from transformers import pipeline

/Users/arystankamalov/Desktop/Minecraft_RAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
client = PersistentClient(path="./db")

emb_minilm = SentenceTransformer("all-MiniLM-L6-v2")
emb_mpnet  = SentenceTransformer("all-mpnet-base-v2")

col_minilm = client.get_collection("minecraft_minilm")
col_mpnet  = client.get_collection("minecraft_mpnet")

llm_qwen = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct", device=-1)
llm_tiny = pipeline("text-generation", model="HuggingFaceTB/SmolLM2-360M-Instruct", device=-1)

print("models loaded!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11503.48it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2615.33it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 3709.40it/s]


models loaded!


In [4]:
prompt_minimal = """Context: {context}
Question: {question}
Answer:"""

prompt_detailed = """You are a Minecraft expert assistant.
Use ONLY the context below to answer the question.
If the answer is not in the context, say "I don't know."

Context:
{context}

Question: {question}
Answer:"""

prompt_cot = """You are a Minecraft expert.
Think step by step using the context provided.

Context:
{context}

Question: {question}
Let's think step by step:"""

In [5]:
def ask_minecraft(question, embedding_choice, llm_choice, prompt_choice, top_k):

    if embedding_choice == "MiniLM":
        emb_model = emb_minilm
        collection = col_minilm
    else:
        emb_model = emb_mpnet
        collection = col_mpnet

    if llm_choice == "Qwen":
        llm = llm_qwen
    else:
        llm = llm_tiny

    if prompt_choice == "minimal":
        prompt_template = prompt_minimal
    elif prompt_choice == "detailed":
        prompt_template = prompt_detailed
    else:
        prompt_template = prompt_cot

    query_embedding = emb_model.encode([question], normalize_embeddings=True).tolist()
    results = collection.query(query_embeddings=query_embedding, n_results=int(top_k))
    docs = results['documents'][0]
    context = "\n".join([line for line in docs])

    prompt = prompt_template.format(context=context, question=question)

    out = llm(prompt, max_new_tokens=200, do_sample=False, repetition_penalty=1.1, truncation=True)
    answer = out[0]["generated_text"][len(prompt):].strip()

    ctx1 = docs[0] if len(docs) > 0 else ""
    ctx2 = docs[1] if len(docs) > 1 else ""
    ctx3 = docs[2] if len(docs) > 2 else ""

    return answer, ctx1, ctx2, ctx3

In [ ]:
demo = gr.Interface(
    fn=ask_minecraft,
    inputs=[
        gr.Textbox(label="your question", placeholder="How do I craft a diamond sword?"),
        gr.Dropdown(["MiniLM", "MPNet"], value="MiniLM", label="embedding model"),
        gr.Dropdown(["Qwen", "SmolLM"], value="Qwen", label="llm"),
        gr.Dropdown(["minimal", "detailed", "chain-of-thought"], value="detailed", label="prompt template"),
        gr.Slider(1, 5, value=3, step=1, label="top_k"),
    ],
    outputs=[
        gr.Textbox(label="answer", lines=5),
        gr.Textbox(label="context 1", lines=4),
        gr.Textbox(label="context 2", lines=4),
        gr.Textbox(label="context 3", lines=4),
    ],
    title="Minecraft RAG Q&A",
    description="Ask anything about Minecraft!",
    examples=[
        ["How do I craft a sword?", "MiniLM", "Qwen", "detailed", 3],
        ["What does a creeper drop?", "MPNet", "SmolLM", "minimal", 3],
        ["How do I build a nether portal?", "MiniLM", "Qwen", "chain-of-thought", 5],
    ]
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'repetition_penalty', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
